In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import random
from scipy.stats import poisson, norm
from scipy.optimize import least_squares
from sklearn.cluster import KMeans
import time



from StockPathSimulation import StockPathSimulation
from StrategySimulation import StrategySimulation

### Parameter Optimization on Simulated Data

In [3]:
t = 50/252
nSims = 10
nSteps = 50

sps = StockPathSimulation(expirationTime = t,
                          numOfSims = nSims,
                          numOfSteps = nSteps)

interestRate = 0.01

ss = StrategySimulation(expirationTime = t,
                        numOfSims = nSims,
                        numOfSteps = nSteps,
                        interestRate = interestRate)

initialPrice = 100.0
strikePrices = np.array([80.0, 90.0, 100.0, 110.0, 120.0])

#### Nonlinear Least Squares

We will use the method of nonlinear least squares to find the optimal parameters for three different models. We set somewhat arbitrary but consistent bounds for the parameters and use a random value (within the bounds) as the initial guess for the parameters.

First, we consider the price of a call option when the underlying stock is modeled by geometric Brownian motion.

In [6]:
lowerBounds = np.array([
    .05  # lower bound for volatility
])

upperBounds = np.array([
    .9  # upper bound for volatility
])

trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
trueMeanRateOfReturn = random.uniform(interestRate, 0.4)

process = sps.simGeomBrownianMotion(meanRateOfReturn = trueMeanRateOfReturn,
                                   volatility = trueVolatility,
                                   initialPrice = initialPrice)

optionPrice = ss.kappa(process, strikePrices, trueVolatility)

In [7]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):

    model_prices = ss.kappa(stockPrices,strikePrices, guess)

    return (marketPrices - model_prices).flatten()

In [8]:
guess = random.uniform(lowerBounds[0], upperBounds[0])

result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices,process,optionPrice))


In [9]:
print(f'Message: ' + result['message'])
print(f"Success status: {result['success']}")
print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
print(f"The value of the cost function at the estimated parameter is {result['cost']}")
print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")

Message: `gtol` termination condition is satisfied.
Success status: True
The estimated volatility is 0.8804613201881158 and the true volatility is 0.880461320188116
The value of the cost function at the estimated parameter is 5.39666761292728e-26
Are there any parameters that are at the bounds?: False


Next, we consider the price of a call option when the underlying stock is modeled by
\begin{equation*}
S(t) = S(0) e^{(\alpha - \lambda \sigma)t} (\sigma+1)^{N(t)},
\end{equation*}
where N(t) is a Poisson process with intensity $\lambda > 0$.

In [11]:
error = 1e-6

lowerBounds = np.array([
    .05,  # lower bound for volatility
    .9    # lower bound for parameter 
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    31   # upper bound for parameter
])

trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
trueMeanRateOfReturn = random.uniform(.005, .1)
trueIntensity = random.uniform(1.0, 30.0)
trueParam = trueIntensity - ((trueMeanRateOfReturn - ss.r)/trueVolatility)

guess = [
    random.uniform(lowerBounds[0], upperBounds[0]),
    random.uniform(lowerBounds[1], upperBounds[1])
]

process = sps.assetModel1(meanRateOfReturn = trueMeanRateOfReturn,
                          volatility = trueVolatility,
                          intensity = trueIntensity,
                          initialPrice = initialPrice)

optionPrice = ss.poissonCallPrice(process, strikePrices, trueVolatility, trueParam, error)



In [12]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    volatility, ltild = guess
    model_prices = ss.poissonCallPrice(stockPrices, strikePrices, volatility, ltild, error)

    return (marketPrices - model_prices).flatten()

result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, process, optionPrice))

In [13]:
print(f'Message: ' + result['message'])
print(f"Success status: {result['success']}")
print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
print(f"The estimated parameter is {result['x'][1]} and the true parameter is {trueParam}")
print(f"The value of the cost function at the estimated parameter is {result['cost']}")
print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")

Message: `gtol` termination condition is satisfied.
Success status: True
The estimated volatility is 0.7960283996939461 and the true volatility is 0.7960283996939418
The estimated parameter is 27.388895969820364 and the true parameter is 27.388895969820336
The value of the cost function at the estimated parameter is 3.6118961088926775e-23
Are there any parameters that are at the bounds?: False


Finally, we consider the price of a call option when the underlying stock is modeled by
\begin{equation*}
S(t)=S(0)\text{exp}\biggl\{\sigma W(t) + \left( \alpha - \beta\lambda -\frac{1}{2}\sigma^2\right) t \biggr\} \prod_{i=1}^{N(t)}(Y_i+1).
\end{equation*}
where $N(t)$ is a Poisson process with intensity $\lambda$ and $Y_1, Y_2, \dotsc$ form a sequence of independent and identically distributed random variables with mean $\beta = \mathbb{E}(Y_i)$. We further assume that, for each $i$, $Y_i$ takes on finitely many values $y_1, \dotsc, y_M$ and write $p(y_m) = P(\{Y_i = y_m\})$.

We will consider the cases when $M = 2$ and $M = 3$.

In [15]:
lowerBounds = np.array([
    .05,  # lower bound for volatility
    -0.6, # lower bound for jump size
    -0.6,
    0,    # lower bound for parameters
    0
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    1,   # upper bound for jump size
    1,
    10,  # upper bound for parameters
    10
])


trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
trueMeanRateOfReturn = random.uniform(.005, .3)
trueJumps = np.random.uniform(low=lowerBounds[1], high=upperBounds[1], size= 2)
trueParameters = np.random.uniform(low=lowerBounds[3], high=upperBounds[3], size= 2)
while (np.any(trueParameters) == False):
    trueParameters = np.random.uniform(low=lowerBounds[3], high=upperBounds[3], size= 2)




process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                          volatility = trueVolatility,
                          intensity = np.sum(trueParameters),
                          jumps = trueJumps,
                          jumpProbabilities = trueParameters/np.sum(trueParameters),
                          initialPrice = initialPrice)

optionPrice = ss.jumpDiffusionCallPrice(process, strikePrices, trueVolatility, trueJumps, trueParameters, error)

In [16]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    volatility = guess[0]
    jumps = guess[1:3]
    params = guess[3:5]
    model_prices = ss.jumpDiffusionCallPrice(process, strikePrices, volatility, jumps, params, error)
    
    return (marketPrices - model_prices).flatten()

guess = np.random.uniform(low = lowerBounds, high = upperBounds)

start = time.perf_counter()

result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, process, optionPrice), max_nfev=80)

end = time.perf_counter()

In [17]:
print(f'Message: ' + result['message'])
print(f"Success status: {result['success']}")
print(f"The inital guesses are {guess}")
print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
print(f"The estimated jump sizes are {result['x'][1:3]} and the true jump sizes are {trueJumps}")
print(f"The estimated parameters are {result['x'][3:]} and the true parameters are {trueParameters}")
print(f"The value of the cost function at the estimated parameter is {result['cost']}")
print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
print(f'Time for least squares optimization: {end - start} seconds') 

Message: `ftol` termination condition is satisfied.
Success status: True
The inital guesses are [0.40835654 0.00742012 0.60114147 6.1861282  5.6525834 ]
The estimated volatility is 0.8999999999999999 and the true volatility is 0.7357927989888899
The estimated jump sizes are [0.18024927 0.58737787] and the true jump sizes are [ 0.58142486 -0.50622622]
The estimated parameters are [10.         6.9072801] and the true parameters are [8.29954443 0.74659841]
The value of the cost function at the estimated parameter is 18.344965698230595
Are there any parameters that are at the bounds?: True
Time for least squares optimization: 41.801275600038934 seconds


In [18]:
lowerBounds = np.array([
    .05,  # lower bound for volatility
    -0.6, # lower bound for jump size
    -0.6,
    -0.6,
    0,    # lower bound for parameters
    0,
    0
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    1,   # upper bound for jump size
    1,
    1,
    10,  # upper bound for parameters
    10,
    10
])


trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
trueMeanRateOfReturn = random.uniform(.005, .3)
trueJumps = np.random.uniform(low=lowerBounds[1], high=upperBounds[1], size= 3)
trueParameters = np.random.uniform(low=lowerBounds[4], high=upperBounds[4], size= 3)
while (np.any(trueParameters) == False):
    trueParameters = np.random.uniform(low=lowerBounds[4], high=upperBounds[4], size= 3)




process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                          volatility = trueVolatility,
                          intensity = np.sum(trueParameters),
                          jumps = trueJumps,
                          jumpProbabilities = trueParameters/np.sum(trueParameters),
                          initialPrice = initialPrice)

optionPrice = ss.jumpDiffusionCallPrice(process, strikePrices, trueVolatility, trueJumps, trueParameters, error)

In [19]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    volatility = guess[0]
    jumps = guess[1:4]
    params = guess[4:7]
    model_prices = ss.jumpDiffusionCallPrice(process, strikePrices, volatility, jumps, params, error)
    
    return (marketPrices - model_prices).flatten()

guess = np.random.uniform(low = lowerBounds, high = upperBounds)

start = time.perf_counter()

result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, process, optionPrice), max_nfev=50)

end = time.perf_counter()

In [20]:
print(f'Message: ' + result['message'])
print(f"Success status: {result['success']}")
print(f"The inital guesses are {guess}")
print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
print(f"The estimated jump sizes are {result['x'][1:4]} and the true jump sizes are {trueJumps}")
print(f"The estimated parameters are {result['x'][4:]} and the true parameters are {trueParameters}")
print(f"The value of the cost function at the estimated parameter is {result['cost']}")
print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
print(f'Time for least squares optimization: {end - start} seconds') 

Message: `gtol` termination condition is satisfied.
Success status: True
The inital guesses are [ 0.84801563  0.16743633  0.47483868 -0.27887146  8.91083028  9.88963329
  5.82978005]
The estimated volatility is 0.08315613946526922 and the true volatility is 0.08315613946525302
The estimated jump sizes are [ 0.29917026  0.76141054 -0.48086579] and the true jump sizes are [ 0.76141054  0.29917026 -0.48086579]
The estimated parameters are [6.39896203 9.37722998 1.49796225] and the true parameters are [9.37722998 6.39896203 1.49796225]
The value of the cost function at the estimated parameter is 2.45091805331606e-24
Are there any parameters that are at the bounds?: False
Time for least squares optimization: 710.3744004000328 seconds


#### Nonlinear Least Squares with "Good Guesses"

As we can see, the least squares optimization takes a long time to converge even when $M$ is relatively low. To fix this issue, we attempt to make "good guesses" of where the optimal parameters should be before we perform least squares optimization. To do this, we use the ideas from "Econometrics of Testing for Jumps in Financial Economics Using Bipower Variation" by Barndoff-Nielsen and Shephard.

Let
\begin{equation*}
Y(t) = Y(0) + \sigma W(t) + \left ( \alpha - \beta \lambda - \frac{1}{2} \sigma^2 \right )t + \sum_{i=1}^{N(t)} \log(Y_i + 1)
\end{equation*}
denote the log price of the stock $S(t)$. Let $n > 0$ be a positive integer. Then, for any $t > 0$,
\begin{equation*}
a_n = \sum_{j=1}^n \left |Y\left ( \frac{jt}{n}\right ) - Y\left ( \frac{(j-1)t}{n} \right ) \right |^2
\end{equation*}
is a consistent estimator of $\sigma^2 t + \sum_{i=1}^{N(t)} \log(Y_i + 1)^2$ and 
\begin{equation*}
b_n = \frac{\pi}{2} \sum_{j=2}^n \left |Y\left ( \frac{jt}{n}\right ) - Y\left ( \frac{(j-1)t}{n} \right ) \right | \left |Y\left ( \frac{(j-1)t}{n}\right ) - Y\left ( \frac{(j-2)t}{n} \right ) \right |,
\end{equation*}
is a consistent estimator of $\sigma^2 t$ as $n \to \infty$. It follows that, $a_n - b_n$ is a consistent estimator of $\sum_{i=1}^{N(t)} \log(Y_i + 1)^2$ as $n \to \infty$. Moreover, under the null hypothesis that $N(t)$ is identically zero for all $t$,
\begin{equation*}
\frac{1}{\sqrt{\frac{\pi^2}{4} + \pi - 5} } \frac{\sqrt{n}(a_n - b_n)}{\sqrt{t}\sigma^2}
\end{equation*}
converges to the standard normal distribution as $n \to \infty$. We will use these facts to make our "good guess".

Let $\{y_0, y_1, \dotsc, y_{nN}\}$ denote the observed values for $Y\left ( 0 \right ), Y\left ( \frac{1}{252*n} \right ), \dotsc, Y\left ( \frac{N}{252} \right )$. We calculate the sample bi-power variations for each day using the formula
\begin{equation*}
b_{nN,d} = \frac{\pi}{2} \sum\limits_{i=n*d+1}^{n*(d+1)-1} |y_{i+1} - y_i||y_i - y_{i-1}|
\end{equation*}
for each $d = 0, \dotsc, N-1$.

In [25]:
def sampleBiPowerVariation(stock, n):
    logRet = np.log(stock[:,1:]) - np.log(stock[:,:-1])
    
    nRows, nCols = logRet.shape
    dailyLogRet = logRet.reshape(nRows, -1, n)

    res  = 1.570796 * np.abs(dailyLogRet[:,:,1:]) * np.abs(dailyLogRet[:,:,:-1])
    return res.sum(axis=2)

For each $d = 0, \dotsc, N-1$, set
\begin{equation*}
a_{nN,d} = \sum\limits_{t=n*d+1}^{n*(d+1)} |y_i - y_{i-1}|^2.
\end{equation*}
For each $d$, we calculate
\begin{equation*}
\hat{g}_d = \frac{1}{\sqrt{\frac{\pi^2}{4} + \pi - 5} }\frac{\sqrt{n}}{\sqrt{252}} \frac{a_{nN,d} - b_{nN,d}}{b_{nN,d}}
\end{equation*}
We set the jump days to be
\begin{equation*}
\{ d = 0, \dotsc, N-1 : |\hat{g}_d| > 2.326\}.
\end{equation*}
On each jump day, we will view a change in price $|y_{i+1} - y_i|$ as a jump if
\begin{equation*}
|y_{i+1} - y_i| > 4\sqrt{b_{nN,d} / n}.
\end{equation*}
Here, we are using the approximation $4\sqrt{b_{nN,d} / n} \approx 4 * \frac{\sigma}{\sqrt{252 *n}}$.

In [27]:
def jumps(stock, n):
    logRet = np.log(stock[:,1:]) - np.log(stock[:,:-1])
    nRows, nCols = logRet.shape
    dailyLogRet = logRet.reshape(nRows, -1, n)
    
    b  = 1.570796 * np.abs(dailyLogRet[:,:,1:]) * np.abs(dailyLogRet[:,:,:-1])
    b = b.sum(axis=2)
    
    a = np.sum(dailyLogRet**2, axis=2)

    statistic = 1.281426 * np.sqrt(n/252) * (a-b)/b
    jumpMask = statistic > 2.326
    jumpMask = np.repeat(jumpMask[:,:,np.newaxis], n, axis = 2)
        
    b = np.repeat(b[:,:,np.newaxis], n, axis = 2)
    maxMask = (np.abs(dailyLogRet) - b)/np.sqrt(b/n) > 4 

    mask = np.logical_and(maxMask, jumpMask)
    return dailyLogRet[mask]




Finally, we perform a clustering analysis on the observed jumps to determine the jump sizes and the jump probabilities of the stock path.

In [29]:
def jumpCluster(stock, n, M):
    js = jumps(stock, n)
    
    js = js.reshape(-1, 1)
    
    kmeans = KMeans(n_clusters = M, n_init = 'auto')
    labels = kmeans.fit_predict(js)
    
    sizes = kmeans.cluster_centers_.flatten()
    probs = np.array([(labels == k).mean() for k in range(M)])
    
    order = np.argsort(sizes)
    sizes = sizes[order]
    probs = probs[order]
    
    return np.exp(sizes)-1, probs

Now, we use the functions we defined above to create our guesses when performing parameter optimization. We will look at call option prices and the prices of the underlying stocks 110 days before expiry. We will look at the stock prices of the first 60 days to create our initial guess for parameter optimization. Then, we will optimize our parameters on the last 50 days of call option prices and underlying stock prices.

In [31]:
t = 110/252
nSims = 10
nSteps = 390 * 110

sps = StockPathSimulation(expirationTime = t,
                          numOfSims = nSims,
                          numOfSteps = nSteps)

lowerBounds = np.array([
    .05,  # lower bound for volatility
    -0.6, # lower bound for jump size
    -0.6,
    0,    # lower bound for parameters
    0
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    1,   # upper bound for jump size
    1,
    10,  # upper bound for parameters
    10
])


trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
trueMeanRateOfReturn = random.uniform(.005, .3)
trueJumps = np.random.uniform(low=lowerBounds[1], high=upperBounds[1], size= 2)
trueParameters = np.random.uniform(low=lowerBounds[3], high=upperBounds[3], size= 2)
while (np.any(trueParameters) == False):
    trueParameters = np.random.uniform(low=lowerBounds[3], high=upperBounds[3], size= 2)




process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                          volatility = trueVolatility,
                          intensity = np.sum(trueParameters),
                          jumps = trueJumps,
                          jumpProbabilities = trueParameters/np.sum(trueParameters),
                          initialPrice = initialPrice)


guessData = process[:,: 390*60 + 1]

optimizationData = process[:,390*60::390]


In [32]:
t = 50/252
nSims = 10
nSteps = 50


interestRate = 0.01

ss = StrategySimulation(expirationTime = t,
                        numOfSims = nSims,
                        numOfSteps = nSteps,
                        interestRate = interestRate)

initialPrice = 100.0
strikePrices = np.array([80.0, 90.0, 100.0, 110.0, 120.0])

optionPrice = ss.jumpDiffusionCallPrice(optimizationData, strikePrices, trueVolatility, trueJumps, trueParameters, error)

In [33]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    volatility = guess[0]
    jumps = guess[1:3]
    params = guess[3:5]
    model_prices = ss.jumpDiffusionCallPrice(stockPrices, strikePrices, volatility, jumps, params, error)
    
    return (marketPrices - model_prices).flatten()

volatility = np.mean(np.sqrt(sampleBiPowerVariation(guessData, 390)/(60/252)))

intensity =  len(jumps(guessData,390))/(10*60/252)

sizes, probs = jumpCluster(guessData, 390, 2)

guess = np.array([
    volatility,
    sizes[0],
    sizes[1],
    probs[0] * intensity,
    probs[1] * intensity
])

start = time.perf_counter()

result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, optimizationData, optionPrice), max_nfev=80)

end = time.perf_counter()

C:\Users\Shin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1436: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [34]:
print(f'Message: ' + result['message'])
print(f"Success status: {result['success']}")
print(f"The inital guesses are {guess}")
print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
print(f"The estimated jump sizes are {result['x'][1:3]} and the true jump sizes are {trueJumps}")
print(f"The estimated parameters are {result['x'][3:]} and the true parameters are {trueParameters}")
print(f"The value of the cost function at the estimated parameter is {result['cost']}")
print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
print(f'Time for least squares optimization: {end - start} seconds') 

Message: `gtol` termination condition is satisfied.
Success status: True
The inital guesses are [0.07119776 0.61512872 0.78655055 4.62       5.88      ]
The estimated volatility is 0.5359064073881803 and the true volatility is 0.535906407388572
The estimated jump sizes are [0.61584272 0.78810632] and the true jump sizes are [0.78810632 0.61584272]
The estimated parameters are [5.41482735 3.63158041] and the true parameters are [3.63158041 5.41482735]
The value of the cost function at the estimated parameter is 5.520656588790691e-23
Are there any parameters that are at the bounds?: False
Time for least squares optimization: 51.688713400042616 seconds


There is a pretty big improvement! Let's do the same for the case when $M = 3$.

In [36]:
t = 110/252
nSims = 10
nSteps = 390 * 110

sps = StockPathSimulation(expirationTime = t,
                          numOfSims = nSims,
                          numOfSteps = nSteps)

lowerBounds = np.array([
    .05,  # lower bound for volatility
    -0.6, # lower bound for jump size
    -0.6,
    -0.6,
    0,    # lower bound for parameters
    0,
    0
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    1,   # upper bound for jump size
    1,
    1,
    10,  # upper bound for parameters
    10,
    10
])


trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
trueMeanRateOfReturn = random.uniform(.005, .3)
trueJumps = np.random.uniform(low=lowerBounds[1], high=upperBounds[1], size= 3)
trueParameters = np.random.uniform(low=lowerBounds[4], high=upperBounds[4], size= 3)
while (np.any(trueParameters) == False):
    trueParameters = np.random.uniform(low=lowerBounds[4], high=upperBounds[4], size= 3)




process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                          volatility = trueVolatility,
                          intensity = np.sum(trueParameters),
                          jumps = trueJumps,
                          jumpProbabilities = trueParameters/np.sum(trueParameters),
                          initialPrice = initialPrice)

guessData = process[:,: 390*60 + 1]

optimizationData = process[:,390*60::390]


In [37]:
t = 50/252
nSims = 10
nSteps = 50


interestRate = 0.01

ss = StrategySimulation(expirationTime = t,
                        numOfSims = nSims,
                        numOfSteps = nSteps,
                        interestRate = interestRate)

initialPrice = 100.0
strikePrices = np.array([80.0, 90.0, 100.0, 110.0, 120.0])

optionPrice = ss.jumpDiffusionCallPrice(optimizationData, strikePrices, trueVolatility, trueJumps, trueParameters, error)

In [41]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    volatility = guess[0]
    jumps = guess[1:4]
    params = guess[4:]
    model_prices = ss.jumpDiffusionCallPrice(stockPrices, strikePrices, volatility, jumps, params, error)
    
    return (marketPrices - model_prices).flatten()

volatility = np.mean(np.sqrt(sampleBiPowerVariation(guessData, 390)/(60/252)))

intensity =  len(jumps(guessData,390))/(10*60/252)

sizes, probs = jumpCluster(guessData, 390, 3)

guess = np.array([
    volatility,
    sizes[0],
    sizes[1],
    sizes[2],
    probs[0] * intensity,
    probs[1] * intensity,
    probs[2] * intensity
])

for i in range(len(guess)):
    if guess[i] < lowerBounds[i]:
        guess[i] = lowerBounds[i]
    if guess[i] > upperBounds[i]:
        guess[i] = upperBounds[i]


start = time.perf_counter()

result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, optimizationData, optionPrice), max_nfev=50)

end = time.perf_counter()

C:\Users\Shin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1436: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [42]:
print(f'Message: ' + result['message'])
print(f"Success status: {result['success']}")
print(f"The inital guesses are {guess}")
print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
print(f"The estimated jump sizes are {result['x'][1:4]} and the true jump sizes are {trueJumps}")
print(f"The estimated parameters are {result['x'][4:]} and the true parameters are {trueParameters}")
print(f"The value of the cost function at the estimated parameter is {result['cost']}")
print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
print(f'Time for least squares optimization: {end - start} seconds') 

Message: `gtol` termination condition is satisfied.
Success status: True
The inital guesses are [ 0.10346186 -0.47978522 -0.31396004  0.45735446  7.14       10.
 10.        ]
The estimated volatility is 0.7641660350028713 and the true volatility is 0.7641660362807401
The estimated jump sizes are [-0.47925867 -0.31400716  0.45770915] and the true jump sizes are [-0.47925867  0.45770915 -0.31400716]
The estimated parameters are [8.75401463 9.92098426 8.86533645] and the true parameters are [8.75401467 8.86533643 9.92098421]
The value of the cost function at the estimated parameter is 4.185696821902246e-17
Are there any parameters that are at the bounds?: False
Time for least squares optimization: 464.8668558000354 seconds


Nonlinear least squares converges at a higher rate than before!